# Filter pages

This notebook identifies which pages actually contain Uno and which don't.

In [1]:
import glob

root_dir = "input/pkna"
output_dir = "output/filtered-pages"
model_name = "gemini-2.5-flash-preview-04-17"
thinking_budget = 1024

all_dirs = sorted(glob.glob(f"{root_dir}/*"))
all_dirs[:10]

['input/pkna/pkna-0',
 'input/pkna/pkna-0-2',
 'input/pkna/pkna-0-3',
 'input/pkna/pkna-1',
 'input/pkna/pkna-10',
 'input/pkna/pkna-11',
 'input/pkna/pkna-12',
 'input/pkna/pkna-13',
 'input/pkna/pkna-14',
 'input/pkna/pkna-15']

In [2]:
import glob


def get_pages(path: str):
    pages = glob.glob(f"{path}/*.jpg") + glob.glob(f"{path}/*.jpeg")
    return sorted(pages)


get_pages(all_dirs[0])[:10]

['input/pkna/pkna-0/pkna-0-000.jpg',
 'input/pkna/pkna-0/pkna-0-001.jpg',
 'input/pkna/pkna-0/pkna-0-002.jpg',
 'input/pkna/pkna-0/pkna-0-003.jpg',
 'input/pkna/pkna-0/pkna-0-004.jpg',
 'input/pkna/pkna-0/pkna-0-005.jpg',
 'input/pkna/pkna-0/pkna-0-006.jpg',
 'input/pkna/pkna-0/pkna-0-007.jpg',
 'input/pkna/pkna-0/pkna-0-008.jpg',
 'input/pkna/pkna-0/pkna-0-009.jpg']

In [3]:
import os
from google import genai
from dotenv import load_dotenv

load_dotenv()
client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))

In [4]:
import json
import time
from google.genai import types
from google.genai.errors import ClientError, ServerError
from pydantic import BaseModel, Field
import PIL.Image

prompt = """The input is a series of pages from a comic book. For each page, provide a true or false answer to the following question: "Does the character Uno appear on this page?".
The character Uno has a duck-like appearance, inside a sphere that appears to be made of a bright green gelatinous substance, with small bubbles. It has a short, rounded beak, large, black eyes without defined pupils.
Do not confuse Uno with "Due", which has a similar appearance but is red.
Provide the answer with exactly one boolean value for each page. Do not skip any pages.
"""


class Response(BaseModel):
    uno_presence: list[bool] = Field(
        description="List indicating presence of Uno on each page"
    )


def identify_uno(pages: list[str]) -> list[bool]:
    images = [PIL.Image.open(page) for page in pages]

    count = 0
    while True:
        try:
            response = client.models.generate_content(
                model=model_name,
                config=types.GenerateContentConfig(
                    response_mime_type="application/json",
                    response_schema=Response.model_json_schema(),
                    thinking_config=types.ThinkingConfig(
                        thinking_budget=thinking_budget
                    ),
                ),
                contents=[prompt] + images,  # type: ignore
            )
            break

        except ClientError as e:
            if e.code != 429 or count > 3:
                raise
            print(f"Rate limit exceeded: {e}, retrying in 30 seconds...")
            time.sleep(30)

        except ServerError as e:
            if count > 3:
                raise
            print(f"Server error: {e}, retrying in 30 seconds...")
            time.sleep(30)

    count = 0
    if not response.text:
        raise ValueError("No response received from the model.")
    jresp = json.loads(response.text)
    return jresp["uno_presence"]

In [ ]:
yes_pages = [f"{root_dir}/pkna-2/pkna2-{x}.jpg" for x in ("07", "09")]
no_pages = [f"{root_dir}/pkna-2/pkna2-{x}.jpg" for x in ("05", "60")]

# Test the function on repeat, with a small set of pages
# This gives us a better idea of the model's confidence
test_pages = yes_pages + no_pages
prob = [0] * len(test_pages)
for _ in range(10):
    resp = identify_uno(test_pages)
    for i, r in enumerate(resp):
        if r:
            prob[i] += 1
prob = [p / 10 for p in prob]
prob

[1.0, 1.0, 0.0, 0.8]

## Run batches

In [ ]:
from tqdm import tqdm
import hashlib

batch_size = 5


def process_directory(dir_path: str):
    # Compute output file name and skip if it already exists.
    prompt_hash = hashlib.sha1(prompt.encode("utf-8")).hexdigest()[:8]
    out_file = os.path.join(
        output_dir, f"{model_name}-{prompt_hash}", os.path.basename(dir_path) + ".json"
    )
    if os.path.exists(out_file):
        print(f"Output file {out_file} already exists. Skipping.")
        return
    os.makedirs(os.path.dirname(out_file), exist_ok=True)

    pages = get_pages(dir_path)
    # Batch in groups to optimize the number of requests
    batches = [pages[i : i + batch_size] for i in range(0, len(pages), batch_size)]

    results = {
        "prompt": prompt,
        "model": model_name,
        "input_pages": pages,
        "uno_presence": [],
        "unknown_presence": [],
    }

    for batch in tqdm(batches, desc=f"Processing batches of {dir_path}"):
        try:
            presence = identify_uno(batch)
        except ValueError as e:
            print(f"Error in response: {e}")
            results["unknown_presence"] += batch
            continue

        except:
            print(f"Error processing batch: {batch}")
            # Save partial results
            with open(out_file.replace(".json", ".part.json"), "w") as f:
                json.dump(results, f, indent=2, ensure_ascii=False)
            raise

        if len(presence) != len(batch):
            print(f"Warning: Expected {len(batch)} responses, got {len(presence)}")
        presence += [True] * (len(batch) - len(presence))  # Pad with True if needed
        results["uno_presence"] += [
            page for page, presence in zip(batch, presence) if presence
        ]

    with open(out_file, "w") as f:
        json.dump(results, f, indent=2, ensure_ascii=False)

In [7]:
process_directory(all_dirs[0])

Output file output/filtered-pages/gemini-2.5-flash-preview-04-17-bd430ff0/pkna-0.json already exists. Skipping.


In [8]:
for dp in all_dirs:
    process_directory(dp)

Output file output/filtered-pages/gemini-2.5-flash-preview-04-17-bd430ff0/pkna-0.json already exists. Skipping.
Output file output/filtered-pages/gemini-2.5-flash-preview-04-17-bd430ff0/pkna-0-2.json already exists. Skipping.
Output file output/filtered-pages/gemini-2.5-flash-preview-04-17-bd430ff0/pkna-0-3.json already exists. Skipping.
Output file output/filtered-pages/gemini-2.5-flash-preview-04-17-bd430ff0/pkna-1.json already exists. Skipping.
Output file output/filtered-pages/gemini-2.5-flash-preview-04-17-bd430ff0/pkna-10.json already exists. Skipping.
Output file output/filtered-pages/gemini-2.5-flash-preview-04-17-bd430ff0/pkna-11.json already exists. Skipping.
Output file output/filtered-pages/gemini-2.5-flash-preview-04-17-bd430ff0/pkna-12.json already exists. Skipping.
Output file output/filtered-pages/gemini-2.5-flash-preview-04-17-bd430ff0/pkna-13.json already exists. Skipping.
Output file output/filtered-pages/gemini-2.5-flash-preview-04-17-bd430ff0/pkna-14.json already e

Processing batches of input/pkna/pkna-35:  77%|███████▋  | 10/13 [00:36<00:11,  3.67s/it]

Rate limit exceeded: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerMinutePerProjectPerModel-FreeTier', 'quotaDimensions': {'location': 'global', 'model': 'gemini-2.5-flash'}, 'quotaValue': '10'}]}, {'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.RetryInfo', 'retryDelay': '59s'}]}}, retrying in 30 seconds...


Processing batches of input/pkna/pkna-36:  62%|██████▏   | 8/13 [00:31<00:20,  4.17s/it]

Rate limit exceeded: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerMinutePerProjectPerModel-FreeTier', 'quotaDimensions': {'location': 'global', 'model': 'gemini-2.5-flash'}, 'quotaValue': '10'}]}, {'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.RetryInfo', 'retryDelay': '43s'}]}}, retrying in 30 seconds...


...

Processing batches of input/pkna/pkna-8:  38%|███▊      | 5/13 [00:17<00:27,  3.40s/it]

Rate limit exceeded: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerMinutePerProjectPerModel-FreeTier', 'quotaDimensions': {'location': 'global', 'model': 'gemini-2.5-flash'}, 'quotaValue': '10'}]}, {'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.RetryInfo', 'retryDelay': '4s'}]}}, retrying in 30 seconds...


Processing batches of input/pkna/pkna-9:  23%|██▎       | 3/13 [00:10<00:34,  3.49s/it]

Rate limit exceeded: 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerMinutePerProjectPerModel-FreeTier', 'quotaDimensions': {'location': 'global', 'model': 'gemini-2.5-flash'}, 'quotaValue': '10'}]}, {'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.RetryInfo', 'retryDelay': '45s'}]}}, retrying in 30 seconds...


Processing batches of input/pkna/pkna-9: 100%|██████████| 13/13 [01:21<00:00,  6.24s/it]
